# Experiment 001-PKV Modular Colab Notebook

本 notebook 用于在 Colab 上复现 Stage 1 的 Experiment 001-PKV 实验。

特点：

- 使用 `edge_llm_systems/` 中的可复用模块；
- 只保留 3-run average + extended sequence 的 PKV 测量流程；
- 不包含旧 v1 单次测量流程；
- 输出文件使用 `_modular` 后缀，不覆盖既有 PKV 结果。

## 1. Mount Google Drive and Locate Project

In [ ]:
# Colab runtime is temporary, so we mount Google Drive to access the repo,
# cache model weights, and persist experiment outputs.
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount('/content/drive')

# If your Drive path changes, this is the only project-root path to update.
PROJECT_DIR = Path('/content/drive/MyDrive/EdgeLLM-Systems')
MODEL_DIR = PROJECT_DIR / 'models'
RESULT_DIR = PROJECT_DIR / 'results' / 'exp001'
TEMP_DIR = RESULT_DIR / 'temp'
CSV_DIR = TEMP_DIR / 'csv'
FIGURE_DIR = TEMP_DIR / 'figures'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Add the repo root to Python's import path so Colab can import local modules
# such as edge_llm_systems, benchmarks, and scripts.
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f'PROJECT_DIR: {PROJECT_DIR}')
print(f'MODEL_DIR:   {MODEL_DIR}')
print(f'RESULT_DIR:  {RESULT_DIR}')
print(f'CSV_DIR:     {CSV_DIR}')
print(f'FIGURE_DIR:  {FIGURE_DIR}')

## 2. Install Dependencies

In [ ]:
# Install the minimal runtime needed by the modular benchmark and plotting code.
!pip install -q torch transformers accelerate sentencepiece huggingface_hub pyyaml pandas matplotlib

## 3. Hugging Face Authentication

In [ ]:
# Gemma models usually require a Hugging Face token with accepted access terms.
# Store it in Colab Secrets as HF_TOKEN.
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    print('HF login success')
else:
    print('HF_TOKEN not found in Colab userdata. If the model requires access, please add it first.')

## 4. Import Modular Benchmark Code

In [ ]:
# This cell imports reusable code instead of redefining benchmark functions here.
# The notebook remains the interactive entrypoint, while logic lives in modules.
import torch

from benchmarks.profiling.run_exp001_profile import (
    average_records,
    build_configs,
    dtype_from_config,
    load_config,
    measure_single,
    write_csv,
)
from edge_llm_systems.cuda_utils import cleanup_cuda, require_cuda
from edge_llm_systems.models import load_causal_lm
from scripts.plot_exp001 import plot_exp001

# Fail early if Colab is not connected to a GPU runtime.
require_cuda()
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('PyTorch:', torch.__version__)

## 5. Experiment Config

从 `benchmarks/configs/exp001_gemma2_t4.yaml` 读取标准配置。

Notebook 作为 Colab 入口，默认将新运行结果写入 `results/exp001/temp/`，避免覆盖本地已确认的正式结果：

- `results/exp001/temp/csv/exp001_results_pkv_modular.csv`
- `results/exp001/temp/figures/exp001_profiling_results_pkv_modular.png`

In [ ]:
# Read the standard Experiment 001 config used by both notebook and CLI flows.
CONFIG_PATH = PROJECT_DIR / 'benchmarks' / 'configs' / 'exp001_gemma2_t4.yaml'
config = load_config(CONFIG_PATH)

# Colab notebook overrides: use mounted Drive paths and write fresh runs to temp.
config['experiment_name'] = 'exp001_gemma2_t4_pkv_modular'
config['model_dir'] = str(MODEL_DIR)
config['output_csv'] = str(CSV_DIR / 'exp001_results_pkv_modular.csv')
config['output_figure'] = str(FIGURE_DIR / 'exp001_profiling_results_pkv_modular.png')

# Build the final prompt_len/gen_len matrix after loading config overrides.
all_configs = build_configs(config)
print(f'Config loaded: {CONFIG_PATH}')
print(f'Total benchmark configs: {len(all_configs)}')
print(all_configs)

## 6. Load Model

In [ ]:
# Load tokenizer and model once, then reuse them for warm-up and all measured runs.
# load_causal_lm will reuse MODEL_DIR if the model has already been downloaded.
tokenizer, model, model_path = load_causal_lm(
    model_id=config['model_id'],
    model_dir=config['model_dir'],
    torch_dtype=dtype_from_config(config['torch_dtype']),
    device_map=config['device_map'],
)
# device_map='auto' decides placement; this records where the model actually lives.
device = next(model.parameters()).device

print(f'Model loaded: {config["model_id"]}')
print(f'Model path: {model_path}')
print(f'Device: {device}')

## 7. Warm-up

覆盖所有 prompt length，`gen_len` 统一使用 64，减少不同 prompt length 首次执行污染正式测量。

In [ ]:
# Warm-up runs are not saved. They compile/initialize kernel paths before timing.
print('Warming up GPU...')
for prompt_len, gen_len in config['warmup_configs']:
    try:
        # measure_single performs one complete prompt_len/gen_len profile.
        _ = measure_single(
            prompt_len,
            gen_len,
            model,
            tokenizer,
            device,
            config['base_prompt'],
        )
        print(f'  warm-up done: prompt={prompt_len}, gen={gen_len}')
    except torch.cuda.OutOfMemoryError:
        cleanup_cuda()
        print(f'  warm-up OOM skipped: prompt={prompt_len}, gen={gen_len}')
print('Warm-up done.')

## 8. Run 3-Run Average PKV Benchmark

In [ ]:
# rows will hold one averaged CSV row per prompt_len/gen_len configuration.
rows = []
repeat = int(config['repeat'])

print('Running Experiment 001-PKV modular benchmark...')
print(f"{'prompt':>8} {'gen':>6} {'TTFT(ms)':>10} {'TPOT(ms)':>10} "
      f"{'tok/s':>8} {'peak(MB)':>10} {'kv_est(MB)':>11} "
      f"{'pkv_pre(MB)':>12} {'pkv_final(MB)':>13}")
print('-' * 108)

# Formal benchmark loop: run each config repeat times, then average the metrics.
for prompt_len, gen_len in all_configs:
    raw = []
    for repeat_idx in range(repeat):
        try:
            # Each raw record contains TTFT, TPOT, peak memory, and PKV metrics.
            raw.append(
                measure_single(
                    prompt_len,
                    gen_len,
                    model,
                    tokenizer,
                    device,
                    config['base_prompt'],
                )
            )
        except torch.cuda.OutOfMemoryError:
            cleanup_cuda()
            print(f'OOM skipped: prompt={prompt_len}, gen={gen_len}, repeat={repeat_idx + 1}')
            break

    if not raw:
        continue

    # Convert repeated raw measurements into one stable result row.
    row = average_records(raw)
    rows.append(row)
    print(f"{row['prompt_len']:>8} {row['gen_len']:>6} {row['ttft_ms']:>10} "
          f"{row['tpot_ms']:>10} {row['tokens_s']:>8} "
          f"{row['peak_mem_mb']:>10} {row['kv_est_mb']:>11} "
          f"{row['kv_pkv_prefill_mb']:>12} {row['kv_pkv_final_mb']:>13}")

print(f'Done. Total completed configs: {len(rows)}')

## 9. Save CSV

In [ ]:
# Persist the modular result under a separate filename so canonical PKV results
# are not overwritten.
from pathlib import Path

csv_path = Path(config['output_csv'])
write_csv(csv_path, rows)
print(f'CSV saved: {csv_path}')

## 10. Plot Results

In [ ]:
# Generate the overview figure from the saved CSV. Plotting is separated from
# measurement so the same CSV can be replotted without rerunning the model.
figure_path = Path(config['output_figure'])
plot_exp001(csv_path, figure_path)
print(f'Figure saved: {figure_path}')